# This notebook is to aggrgate results to administrative boundaries

In [ ]:
import warnings
warnings.simplefilter('ignore', FutureWarning)
import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr
import shapely
import igraph as ig
from pathlib import Path
from exactextract import exact_extract
import geohexgrid as ghg
from shapely.strtree import STRtree
import pyarrow 

import matplotlib.pyplot as plt
import contextily as cx
from matplotlib.colors import ListedColormap


: 

In [2]:
root_dir = Path(r"P:\11212759-001-brs\002-Kaartverhal_KEA\MS2\MS2proposal\data\paramaribo\outputs_ver04")
network_path = root_dir/"data"/"static" / "network"
output_path = root_dir / "data" / "static" / "output_graph"

workfolder_test = root_dir /"outputs"
base = root_dir/"outputs"
workfolder_test.mkdir(parents=True, exist_ok=True)

In [3]:
# results

driving_combined_path = workfolder_test / "combined" /"driving_combined_results.gpkg"
driving_combined = gpd.read_file(driving_combined_path)
print(driving_combined.columns)

walking_combined_path = workfolder_test / "combined" /"walking_combined_results.gpkg"
walking_combined = gpd.read_file(walking_combined_path)
print(walking_combined.columns)


Index(['cell_id', 'population', 'dist_to_nearest_road_m', 'node',
       'time_nearest_supermarket_min', 'distance_nearest_supermarket_km',
       'connected_supermarket', 'time_nearest_school_min',
       'distance_nearest_school_km', 'connected_school',
       'time_nearest_highway_bus_stop_min',
       'distance_nearest_highway_bus_stop_km', 'connected_highway_bus_stop',
       'time_nearest_hospital_min', 'distance_nearest_hospital_km',
       'connected_hospital', 'HZ_population', 'HZ_dist_to_nearest_road_m',
       'HZ_node', 'HZ_time_nearest_supermarket_min',
       'HZ_distance_nearest_supermarket_km', 'HZ_connected_supermarket',
       'HZ_time_nearest_school_min', 'HZ_distance_nearest_school_km',
       'HZ_connected_school', 'HZ_time_nearest_highway_bus_stop_min',
       'HZ_distance_nearest_highway_bus_stop_km',
       'HZ_connected_highway_bus_stop', 'HZ_time_nearest_hospital_min',
       'HZ_distance_nearest_hospital_km', 'HZ_connected_hospital',
       'FL_population', '

In [4]:
# calculating difference in travel times between baseline and hazard scenarios for each destination category

def build_driving_combined_and_filtered(
    combined_gdf,
    categories=("supermarket", "school", "highway_bus_stop", "hospital"),
):
    combined = combined_gdf.copy()
    flag_columns = []

    for category in categories:
        hz_column = f"HZ_time_nearest_{category}_min"
        base_column = f"time_nearest_{category}_min"
        dif_column = f"dif_time_{category}"
        flag_column = f"flag_very_negative_dif_time_{category}"

        if hz_column not in combined.columns or base_column not in combined.columns:
            raise KeyError(
                f"Missing columns for category '{category}': {hz_column} and/or {base_column}"
            )

        combined[dif_column] = combined[hz_column] - combined[base_column]
        # flag suspiciously large negative differences before clipping them away
        combined[flag_column] = combined[dif_column] < -2
        flag_columns.append(flag_column)
        # negative differences mean the hazard scenario is faster, which shouldn't happen - clip to 0
        combined[dif_column] = combined[dif_column].clip(lower=0)

    combined["flag_very_negative_dif_time"] = combined[flag_columns].any(axis=1)

    required_columns = ["cell_id", "population", "geometry"]
    missing_columns = [column for column in required_columns if column not in combined.columns]
    if missing_columns:
        raise KeyError(f"Missing required columns in combined_gdf: {missing_columns}")

    dif_time_columns = [column for column in combined.columns if column.startswith("dif_time_")]
    selected_columns = [
        "cell_id",
        "population",
        *dif_time_columns,
        *flag_columns,
        "flag_very_negative_dif_time",
        "geometry",
    ]

    filtered_combined = gpd.GeoDataFrame(
        combined[selected_columns].copy(),
        geometry="geometry",
        crs=combined.crs,
    )

    return combined, filtered_combined

driving_combined, driving_combined_filtered = build_driving_combined_and_filtered(
    driving_combined
)
display(driving_combined_filtered.head())
#driving_combined_filtered.to_file(workfolder_test / "combined" / "driving_travel_time_difference.gpkg", driver="GPKG")


walking_combined, walking_combined_filtered = build_driving_combined_and_filtered(
    walking_combined    
)
display(walking_combined_filtered.head())
#walking_combined_filtered.to_file(workfolder_test / "combined" / "walking_travel_time_difference.gpkg", driver="GPKG")


,cell_id,population,dif_time_supermarket,dif_time_school,dif_time_highway_bus_stop,dif_time_hospital,flag_very_negative_dif_time_supermarket,flag_very_negative_dif_time_school,flag_very_negative_dif_time_highway_bus_stop,flag_very_negative_dif_time_hospital,flag_very_negative_dif_time,geometry
0,"9371,14731",4.524825,0.298698,0.216884,2.098105,4.949630,False,False,False,False,False,"POLYGON ((702800 637827.71, 702775 637871.011,..."
1,"9373,14731",5.094980,0.335303,0.180279,2.134710,4.986235,False,False,False,False,False,"POLYGON ((702950 637827.71, 702925 637871.011,..."
2,"9374,14730",5.149691,0.346851,0.168731,2.146258,4.997784,False,False,False,False,False,"POLYGON ((703025 637784.409, 703000 637827.71,..."
3,"9367,14733",4.882089,0.217693,0.226986,2.108206,4.959731,False,False,False,False,False,"POLYGON ((702500 637914.312, 702475 637957.614..."
4,"9369,14733",5.338606,0.243068,0.232158,2.082831,4.934356,False,False,False,False,False,"POLYGON ((702650 637914.312, 702625 637957.614..."


,cell_id,population,dif_time_supermarket,dif_time_school,dif_time_highway_bus_stop,dif_time_hospital,flag_very_negative_dif_time_supermarket,flag_very_negative_dif_time_school,flag_very_negative_dif_time_highway_bus_stop,flag_very_negative_dif_time_hospital,flag_very_negative_dif_time,geometry
0,"9371,14731",4.524825,0.0,0.0,0.0,0.477491,False,False,False,False,False,"POLYGON ((702800 637827.71, 702775 637871.011,..."
1,"9373,14731",5.094980,0.0,0.0,0.0,0.477491,False,False,False,False,False,"POLYGON ((702950 637827.71, 702925 637871.011,..."
2,"9374,14730",5.149691,0.0,0.0,0.0,0.477491,False,False,False,False,False,"POLYGON ((703025 637784.409, 703000 637827.71,..."
3,"9367,14733",4.882089,0.0,0.0,0.0,0.477491,False,False,False,False,False,"POLYGON ((702500 637914.312, 702475 637957.614..."
4,"9369,14733",5.338606,0.0,0.0,0.0,0.477491,False,False,False,False,False,"POLYGON ((702650 637914.312, 702625 637957.614..."


In [8]:
join_keys = ["cell_id", "population"]

driving_for_join = driving_combined_filtered.drop(columns="geometry", errors="ignore")
walking_for_join = walking_combined_filtered.drop(columns="geometry", errors="ignore")

combined_joined = driving_for_join.merge(
    walking_for_join,
    on=join_keys,
    how="outer",
    indicator=True,
    suffixes=("_driving", "_walking"),
)

not_joined_driving_count = int((combined_joined["_merge"] == "left_only").sum())
not_joined_walking_count = int((combined_joined["_merge"] == "right_only").sum())
not_joined_total_count = not_joined_driving_count + not_joined_walking_count

print(f"Rows from driving not joined: {not_joined_driving_count}")
print(f"Rows from walking not joined: {not_joined_walking_count}")
print(f"Total rows not joined: {not_joined_total_count}")

display(combined_joined.head())

Rows from driving not joined: 0
Rows from walking not joined: 0
Total rows not joined: 0


,cell_id,population,dif_time_supermarket_driving,dif_time_school_driving,dif_time_highway_bus_stop_driving,dif_time_hospital_driving,flag_very_negative_dif_time_supermarket_driving,flag_very_negative_dif_time_school_driving,flag_very_negative_dif_time_highway_bus_stop_driving,flag_very_negative_dif_time_hospital_driving,...,dif_time_supermarket_walking,dif_time_school_walking,dif_time_highway_bus_stop_walking,dif_time_hospital_walking,flag_very_negative_dif_time_supermarket_walking,flag_very_negative_dif_time_school_walking,flag_very_negative_dif_time_highway_bus_stop_walking,flag_very_negative_dif_time_hospital_walking,flag_very_negative_dif_time_walking,_merge
0,"9258,14960",0.080951,NaN,NaN,NaN,NaN,False,False,False,False,...,NaN,NaN,NaN,NaN,False,False,False,False,False,both
1,"9258,14962",0.419720,NaN,NaN,NaN,NaN,False,False,False,False,...,NaN,NaN,NaN,NaN,False,False,False,False,False,both
2,"9258,14964",1.201686,0.844288,1.778496,3.142093,3.977131,False,False,False,False,...,0.0,0.244121,0.0,0.0,False,False,False,False,False,both
3,"9258,14966",2.320324,0.760831,1.695039,3.058636,3.893674,False,False,False,False,...,0.0,0.244121,0.0,0.0,False,False,False,False,False,both
4,"9258,14968",3.638126,0.768198,1.702406,3.066003,3.901041,False,False,False,False,...,0.0,0.244121,0.0,0.0,False,False,False,False,False,both


In [9]:
print(combined_joined.columns)

Index(['cell_id', 'population', 'dif_time_supermarket_driving',
       'dif_time_school_driving', 'dif_time_highway_bus_stop_driving',
       'dif_time_hospital_driving',
       'flag_very_negative_dif_time_supermarket_driving',
       'flag_very_negative_dif_time_school_driving',
       'flag_very_negative_dif_time_highway_bus_stop_driving',
       'flag_very_negative_dif_time_hospital_driving',
       'flag_very_negative_dif_time_driving', 'dif_time_supermarket_walking',
       'dif_time_school_walking', 'dif_time_highway_bus_stop_walking',
       'dif_time_hospital_walking',
       'flag_very_negative_dif_time_supermarket_walking',
       'flag_very_negative_dif_time_school_walking',
       'flag_very_negative_dif_time_highway_bus_stop_walking',
       'flag_very_negative_dif_time_hospital_walking',
       'flag_very_negative_dif_time_walking', '_merge'],
      dtype='object')


In [6]:
ratio_driving = 0.5
ratio_walking = 0.5

categories = ["supermarket", "school", "highway_bus_stop", "hospital"]

for category in categories:
    driving_col = f"dif_time_{category}_driving"
    walking_col = f"dif_time_{category}_walking"
    pop_loss_col = f"pop_loss_hours_{category}"

    if driving_col not in combined_joined.columns or walking_col not in combined_joined.columns:
        raise KeyError(f"Missing columns for {category}: {driving_col} and/or {walking_col}")

    weighted_dif_time_minutes = (
        ratio_driving * combined_joined[driving_col].fillna(0)
        + ratio_walking * combined_joined[walking_col].fillna(0)
    )
    weighted_dif_time_hours = weighted_dif_time_minutes / 60
    combined_joined[pop_loss_col] = combined_joined["population"] * weighted_dif_time_hours

display(
    combined_joined[
        [
            "cell_id",
            "population",
            *[f"pop_loss_hours_{category}" for category in categories],
        ]
    ].head()
)

,cell_id,population,pop_loss_hours_supermarket,pop_loss_hours_school,pop_loss_hours_highway_bus_stop,pop_loss_hours_hospital
0,"9258,14960",0.080951,0.000000,0.000000,0.000000,0.000000
1,"9258,14962",0.419720,0.000000,0.000000,0.000000,0.000000
2,"9258,14964",1.201686,0.008455,0.020255,0.031465,0.039827
3,"9258,14966",2.320324,0.014711,0.037496,0.059142,0.075288
4,"9258,14968",3.638126,0.023290,0.059014,0.092954,0.118271


In [ ]:
print(combined_joined)
combined_joined.to_file(
    workfolder_test / "combined" / "combined_travel_time_difference.gpkg", driver="GPKG"
)

          cell_id  population  dif_time_supermarket_driving  \
0      9258,14960    0.080951                           NaN   
1      9258,14962    0.419720                           NaN   
2      9258,14964    1.201686                      0.844288   
3      9258,14966    2.320324                      0.760831   
4      9258,14968    3.638126                      0.768198   
...           ...         ...                           ...   
20134  9489,14949    6.036116                      0.487125   
20135  9489,14951    7.646964                      0.520146   
20136  9489,14953    6.904763                      0.580191   
20137  9489,14955    4.279216                      0.638076   
20138  9489,14957    1.411631                      0.647139   

       dif_time_school_driving  dif_time_highway_bus_stop_driving  \
0                          NaN                                NaN   
1                          NaN                                NaN   
2                     1.778496      

In [13]:
#aggregate to admn areas

admn_path = network_path/"resorten.gpkg"
admn = gpd.read_file(admn_path)

In [17]:
admn = admn.reset_index(drop=True)
admn["admn_id"] = admn.index

# combined_joined has no geometry (dropped during merge), so bring hexagon geometry back
hex_geometry_lookup = driving_combined_filtered[["cell_id", "geometry"]].drop_duplicates("cell_id")
combined_with_geom = gpd.GeoDataFrame(
    combined_joined.merge(hex_geometry_lookup, on="cell_id", how="left"),
    geometry="geometry",
    crs=driving_combined_filtered.crs,
)

# intersect hexagons with admin polygons and keep, for each hexagon, only the admin polygon it overlaps most
overlay = gpd.overlay(
    combined_with_geom,
    admn[["admn_id", "geometry"]],
    how="intersection",
    keep_geom_type=False,
)
overlay["overlap_area"] = overlay.geometry.area

best_admn_per_cell = (
    overlay.sort_values("overlap_area", ascending=False)
    .drop_duplicates("cell_id", keep="first")[["cell_id", "admn_id"]]
)

hex_to_admn = combined_joined.merge(best_admn_per_cell, on="cell_id", how="left")

pop_loss_columns = [column for column in hex_to_admn.columns if column.startswith("pop_loss_hours_")]
admn_aggregated = (
    hex_to_admn.groupby("admn_id")
    .agg(hex_population=("population", "sum"), **{column: (column, "sum") for column in pop_loss_columns})
    .reset_index()
)

admn_results = admn.merge(admn_aggregated, on="admn_id", how="left")
display(admn_results.head())


,OBJECTID,ID,RES_NM,DISTR_NM,wgs84_WGS8,wgs84_WG_1,Shape_Leng,Shape_Area,Population,Surface_Ar,BuurtRes,geometry,admn_id,hex_population,pop_loss_hours_supermarket,pop_loss_hours_school,pop_loss_hours_highway_bus_stop,pop_loss_hours_hospital
0,16.0,0.0,Houttuin,Wanica,0.0,0.0,46678.402050,5.702762e+07,10227,SWM,Wanica Zuid-Oost,"MULTIPOLYGON (((705307.93 640002.305, 705296.1...",0,NaN,NaN,NaN,NaN,NaN
1,19.0,0.0,Latour,Paramaribo,0.0,0.0,13422.028115,7.163114e+06,26148,SWM,Paramaribo Zuid-West,"MULTIPOLYGON (((697032.72 640521, 696853.99 64...",1,29695.932929,676.470250,252.487459,398.449059,2046.359147
2,21.0,0.0,Beekhuizen,Paramaribo,0.0,0.0,11825.678052,6.311734e+06,19783,SWM,Paramaribo Noord-Oost,"MULTIPOLYGON (((700611.51 643051.42, 700809 64...",2,12770.819148,56.568422,74.584313,90.858775,605.283573
3,22.0,0.0,Flora,Paramaribo,0.0,0.0,10021.395951,4.479517e+06,15346,SWM,Paramaribo Zuid-West,"MULTIPOLYGON (((699325.13 644382.51, 699337.8 ...",3,18205.337399,186.332362,120.483414,249.608768,650.682037
4,24.0,0.0,Tammenga,Paramaribo,0.0,0.0,11824.136872,6.315662e+06,14313,SWM,Paramaribo Zuid-West,"MULTIPOLYGON (((696697.22 641756.25, 696623.93...",4,35182.646750,144.354593,120.580416,798.603675,1434.133962


In [18]:
admn_results.to_file(workfolder_test / "combined" / "admn_aggregated_results.gpkg", driver="GPKG")

In [10]:
print(driving_combined.columns)

Index(['cell_id', 'population', 'dist_to_nearest_road_m', 'node',
       'time_nearest_supermarket_min', 'distance_nearest_supermarket_km',
       'connected_supermarket', 'time_nearest_school_min',
       'distance_nearest_school_km', 'connected_school',
       'time_nearest_highway_bus_stop_min',
       'distance_nearest_highway_bus_stop_km', 'connected_highway_bus_stop',
       'time_nearest_hospital_min', 'distance_nearest_hospital_km',
       'connected_hospital', 'HZ_population', 'HZ_dist_to_nearest_road_m',
       'HZ_node', 'HZ_time_nearest_supermarket_min',
       'HZ_distance_nearest_supermarket_km', 'HZ_connected_supermarket',
       'HZ_time_nearest_school_min', 'HZ_distance_nearest_school_km',
       'HZ_connected_school', 'HZ_time_nearest_highway_bus_stop_min',
       'HZ_distance_nearest_highway_bus_stop_km',
       'HZ_connected_highway_bus_stop', 'HZ_time_nearest_hospital_min',
       'HZ_distance_nearest_hospital_km', 'HZ_connected_hospital',
       'FL_population', '

In [14]:
admn_path = network_path/"resorten.gpkg"
admn = gpd.read_file(admn_path)

# Remove an existing polygon-level population attribute; use hex-based population instead.
admn = admn.reset_index(drop=True).drop(columns="population", errors="ignore")
admn["admn_id"] = admn.index

categories = ["supermarket", "school", "highway_bus_stop", "hospital"]

# Ensure both layers use the same CRS before performing spatial operations.
if driving_combined.crs != admn.crs:
    admn = admn.to_crs(driving_combined.crs)

# Assign every hexagon to the administrative polygon with its largest overlap.
hexagons = driving_combined[["cell_id", "population", "geometry"]].drop_duplicates("cell_id").copy()

overlay = gpd.overlay(
    hexagons,
    admn[["admn_id", "geometry"]],
    how="intersection",
    keep_geom_type=False,
)
overlay["overlap_area"] = overlay.geometry.area

best_admn_per_cell = (
    overlay.sort_values("overlap_area", ascending=False)
    .drop_duplicates("cell_id", keep="first")[["cell_id", "admn_id"]]
)

# Attach the selected administrative area to the full driving-results table,
# which contains the baseline and hazard access columns.
hex_to_admn = driving_combined.merge(
    best_admn_per_cell,
    on="cell_id",
    how="left",
)

# Normalise Yes/No values and create population-with-access fields.
access_population_columns = {}

for category in categories:
    baseline_column = f"Access_{category}_baseline"
    hazard_column = f"Access_{category}_hazard"

    missing_columns = [
        column
        for column in [baseline_column, hazard_column]
        if column not in hex_to_admn.columns
    ]
    if missing_columns:
        raise KeyError(
            f"Missing access column(s) for '{category}': {missing_columns}"
        )

    baseline_population_column = f"population_access_{category}_baseline"
    hazard_population_column = f"population_access_{category}_hazard"

    hex_to_admn[baseline_population_column] = np.where(
        hex_to_admn[baseline_column].astype(str).str.strip().str.lower().eq("yes"),
        hex_to_admn["population"],
        0,
    )
    hex_to_admn[hazard_population_column] = np.where(
        hex_to_admn[hazard_column].astype(str).str.strip().str.lower().eq("yes"),
        hex_to_admn["population"],
        0,
    )

    access_population_columns[baseline_population_column] = (
        baseline_population_column,
        "sum",
    )
    access_population_columns[hazard_population_column] = (
        hazard_population_column,
        "sum",
    )

# Aggregate total hex population and population with access by administrative polygon.
admn_aggregated = (
    hex_to_admn.dropna(subset=["admn_id"])
    .groupby("admn_id")
    .agg(
        hex_population=("population", "sum"),
        **access_population_columns,
    )
    .reset_index()
)

admn_results = admn.merge(admn_aggregated, on="admn_id", how="left")

# Administrative areas without assigned hexagons should show zero people.
population_result_columns = ["hex_population", *access_population_columns]
admn_results[population_result_columns] = admn_results[
    population_result_columns
].fillna(0)

display(admn_results)
admn_results.to_file(
    workfolder_test / "combined" / "admn_aggregated_population_accessibility.gpkg", driver="GPKG"
)

,OBJECTID,ID,RES_NM,DISTR_NM,wgs84_WGS8,wgs84_WG_1,Shape_Leng,Shape_Area,Population,Surface_Ar,...,admn_id,hex_population,population_access_supermarket_baseline,population_access_supermarket_hazard,population_access_school_baseline,population_access_school_hazard,population_access_highway_bus_stop_baseline,population_access_highway_bus_stop_hazard,population_access_hospital_baseline,population_access_hospital_hazard
0,16.0,0.0,Houttuin,Wanica,0.0,0.0,46678.402050,5.702762e+07,10227,SWM,...,0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,19.0,0.0,Latour,Paramaribo,0.0,0.0,13422.028115,7.163114e+06,26148,SWM,...,1,29695.932929,29695.932929,29695.932929,29695.932929,29695.932929,29695.932929,29695.932929,29695.932929,29695.932929
2,21.0,0.0,Beekhuizen,Paramaribo,0.0,0.0,11825.678052,6.311734e+06,19783,SWM,...,2,12770.819148,12768.299044,12707.204185,12768.299044,12707.204185,12768.299044,12707.204185,12768.299044,12707.204185
3,22.0,0.0,Flora,Paramaribo,0.0,0.0,10021.395951,4.479517e+06,15346,SWM,...,3,18205.337399,18205.337399,18179.241520,18205.337399,18179.241520,18205.337399,18179.241520,18205.337399,18179.241520
4,24.0,0.0,Tammenga,Paramaribo,0.0,0.0,11824.136872,6.315662e+06,14313,SWM,...,4,35182.646750,35182.646750,35133.386661,35182.646750,35133.386661,35182.646750,35133.386661,35182.646750,35133.386661
5,26.0,0.0,Saramaccapolder,Wanica,0.0,0.0,32037.799365,2.446587e+07,7789,SWM,...,5,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
6,37.0,0.0,Kwatta,Wanica,0.0,0.0,41169.529664,6.684762e+07,10091,SWM,...,6,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
7,52.0,0.0,Meerzorg,Commewijne,0.0,0.0,185724.397856,1.023999e+09,8115,NHDW,...,7,110.086768,106.941458,106.941458,106.941458,106.941458,106.941458,106.941458,106.941458,106.941458
8,57.0,0.0,Nieuw Amsterdam,Commewijne,0.0,0.0,41483.813308,5.881099e+07,5489,NHDW,...,8,161.496772,154.362831,154.362831,154.362831,154.362831,154.362831,154.362831,154.362831,154.362831
9,35.0,0.0,Weg naar Zee,Paramaribo,0.0,0.0,27810.287678,4.349974e+07,13172,SWM,...,9,46408.415184,45427.605867,45427.605867,45427.605867,45427.605867,45427.605867,45427.605867,45427.605867,45427.605867
